In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Iterator, List, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd

import shap

from src.data_prepatation import create_batched_splits

In [ ]:
from config import dir_config
compiled_dir = dir_config.data.compiled
processed_dir = dir_config.data.processed

In [ ]:
target_type = 'apnea'
future_target_step = 1
max_feature_lag = 5

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, classification_report, confusion_matrix

def evaluate_model(model, X, y, threshold=0.5):
    y_pred_proba = model.predict_proba(X)[:, 1]
    y_pred = (y_pred_proba >= threshold).astype(int)
    auc_roc = roc_auc_score(y, y_pred_proba)
    auc_pr = average_precision_score(y, y_pred_proba)
    f1 = f1_score(y, y_pred)
    #
    classification = classification_report(y, y_pred)
    confusion = confusion_matrix(y, y_pred)
    return {'auc_roc': auc_roc, 'auc_pr': auc_pr, 'f1': f1, 'classification_report': classification, 'confusion_matrix': confusion}

# Load data

In [ ]:
# get all parquet files in the processed directory
parquet_files = list(Path(processed_dir).glob('*_with_metadata.parquet'))

# remove "agg_data_with_metadata.parquet" from the list of parquet files
parquet_files = [f for f in parquet_files if f.name != "agg_data_with_metadata.parquet"]


## Chunk in 0.5hr

In [ ]:
train_X, train_y, val_X, val_y, test_X, test_y, left_out = create_batched_splits(
    parquet_files,
    batch_size=360,
    gap_size=6,
    top_features=None,
    top_features_lag=max_feature_lag,
    target_type='apnea',
    target_future_steps=future_target_step,
    val_ratio=0.2,
    test_ratio=0.2,
    n_leave_out=5,
    random_seed=2542,
)

### MLP

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


def prepare_nn_features(train_frame, val_frame, test_frame):
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    train_array = np.asarray(scaler.fit_transform(imputer.fit_transform(train_frame)), dtype=np.float32)
    val_array   = np.asarray(scaler.transform(imputer.transform(val_frame)),   dtype=np.float32)
    test_array  = np.asarray(scaler.transform(imputer.transform(test_frame)),  dtype=np.float32)
    for name, arr in (("train", train_array), ("val", val_array), ("test", test_array)):
        if not np.isfinite(arr).all():
            raise ValueError(f"Non-finite values in {name} features after preprocessing.")
    return train_array, val_array, test_array


class MLP(nn.Module):
    """Residual-style MLP with BatchNorm suited for tabular binary classification."""
    def __init__(self, input_size: int, hidden: int = 256):
        super().__init__()
        def block(in_dim: int, out_dim: int, dropout: float) -> nn.Sequential:
            return nn.Sequential(
                nn.Linear(in_dim, out_dim),
                nn.BatchNorm1d(out_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            )
        self.stem    = block(input_size, hidden, 0.3)
        self.layer1  = block(hidden, hidden // 2, 0.3)
        self.layer2  = block(hidden // 2, hidden // 4, 0.2)
        self.head    = nn.Linear(hidden // 4, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.layer2(self.layer1(self.stem(x))))


def evaluate_torch_model(model, X_tensor, y_tensor, split_name, device, threshold=0.5):
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(X_tensor.to(device)).squeeze(1)).cpu().numpy()
    y_true = y_tensor.squeeze(1).cpu().numpy().astype(int)
    y_pred = (probs >= threshold).astype(int)
    m = {
        "auc_roc":               roc_auc_score(y_true, probs),
        "auc_pr":                average_precision_score(y_true, probs),
        "f1":                    f1_score(y_true, y_pred),
        "classification_report": classification_report(y_true, y_pred),
        "confusion_matrix":      confusion_matrix(y_true, y_pred),
    }
    print(f"{split_name:10s}  AUC-ROC={m['auc_roc']:.4f}  AUC-PR={m['auc_pr']:.4f}  F1={m['f1']:.4f}")
    print(m["classification_report"])
    return m


# ── Data preparation ────────────────────────────────────────────────────────
train_array, val_array, _test_array = prepare_nn_features(train_X, val_X, test_X)

train_features = torch.tensor(train_array, dtype=torch.float32)
val_features   = torch.tensor(val_array,   dtype=torch.float32)

train_targets = torch.tensor(train_y.to_numpy(), dtype=torch.float32).unsqueeze(1)
val_targets   = torch.tensor(val_y.to_numpy(),   dtype=torch.float32).unsqueeze(1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Model / loss ────────────────────────────────────────────────────────────
model = MLP(input_size=train_features.shape[1]).to(device)

neg = float((train_y == 0).sum())
pos = float((train_y == 1).sum())
if pos == 0:
    raise ValueError("No positive examples in training split.")

pos_weight = torch.tensor([neg / pos], dtype=torch.float32, device=device)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer  = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
scheduler  = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=10, min_lr=1e-6
)

train_dataset = TensorDataset(train_features, train_targets)
train_loader  = DataLoader(train_dataset, batch_size=256, shuffle=True)

# ── Training loop with early stopping on val AUC-PR ─────────────────────────
MAX_EPOCHS     = 200
ES_PATIENCE    = 25
best_val_aucpr = -1.0
best_state     = None
epochs_no_imp  = 0

for epoch in range(MAX_EPOCHS):
    model.train()
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        loss = criterion(model(X_batch), y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * X_batch.size(0)

    model.eval()
    with torch.no_grad():
        val_probs = torch.sigmoid(
            model(val_features.to(device)).squeeze(1)
        ).cpu().numpy()
    val_aucpr = float(average_precision_score(val_targets.squeeze(1).numpy().astype(int), val_probs))

    scheduler.step(val_aucpr)

    improved = val_aucpr > best_val_aucpr
    if improved:
        best_val_aucpr = val_aucpr
        best_state     = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_imp  = 0
    else:
        epochs_no_imp += 1

    if (epoch + 1) % 10 == 0 or improved:
        mean_loss = epoch_loss / len(train_dataset)
        marker    = " ✓" if improved else ""
        lr_now    = optimizer.param_groups[0]["lr"]
        print(f"Epoch {epoch+1:>3}/{MAX_EPOCHS}  loss={mean_loss:.4f}  val_AUC-PR={val_aucpr:.4f}  lr={lr_now:.2e}{marker}")

    if epochs_no_imp >= ES_PATIENCE:
        print(f"\nEarly stopping at epoch {epoch + 1} (no improvement for {ES_PATIENCE} epochs).")
        break

# ── Restore best checkpoint and evaluate ────────────────────────────────────
model.load_state_dict(best_state)
print(f"\nBest val AUC-PR: {best_val_aucpr:.4f}")
print("\n─── Final evaluation (train / val) ──────────────────────────────────────")
train_metrics = evaluate_torch_model(model, train_features, train_targets, "Train",      device)
val_metrics   = evaluate_torch_model(model, val_features,   val_targets,   "Validation", device)

In [ ]:
torch.cuda.is_available()

In [ ]:
# is cuda available? if so, move model and data to GPU for faster inference
if torch.cuda.is_available():
    model.to(device)
    train_features = train_features.to(device)
    train_targets  = train_targets.to(device)
    val_features   = val_features.to(device)
    val_targets    = val_targets.to(device)

In [ ]:
# Quick diagnostics for model quality and class balance
print("train metrics:", {k: v for k, v in train_metrics.items() if k in ["auc_roc", "auc_pr", "f1"]})
print("val metrics:", {k: v for k, v in val_metrics.items() if k in ["auc_roc", "auc_pr", "f1"]})

train_pos_rate = float(train_y.mean())
val_pos_rate = float(val_y.mean())
test_pos_rate = float(test_y.mean())
print("pos rates:", {"train": train_pos_rate, "val": val_pos_rate, "test": test_pos_rate})

# Current threshold performance on validation
with torch.no_grad():
    val_probs_diag = torch.sigmoid(model(val_features.to(device)).squeeze(1)).cpu().numpy()
val_pred_diag = (val_probs_diag >= 0.5).astype(int)
val_true_diag = val_targets.squeeze(1).cpu().numpy().astype(int)
print("val predicted positive rate @0.5:", float(val_pred_diag.mean()))
print("val avg predicted prob:", float(val_probs_diag.mean()))

In [ ]:
from sklearn.metrics import precision_recall_curve

# Tune threshold on validation probabilities to maximize F1
val_true = val_targets.squeeze(1).cpu().numpy().astype(int)
with torch.no_grad():
    val_probs_tune = torch.sigmoid(model(val_features.to(device)).squeeze(1)).cpu().numpy()

prec, rec, thr = precision_recall_curve(val_true, val_probs_tune)
# precision_recall_curve returns one extra point for precision/recall without threshold
f1s = 2 * (prec[:-1] * rec[:-1]) / (prec[:-1] + rec[:-1] + 1e-12)
best_idx = int(np.argmax(f1s))
best_thr = float(thr[best_idx])

print("best threshold on val (for F1):", best_thr)
print("best val F1:", float(f1s[best_idx]))

# Compare with default 0.5
val_pred_05 = (val_probs_tune >= 0.5).astype(int)
val_pred_bt = (val_probs_tune >= best_thr).astype(int)
print("val positive rate @0.5:", float(val_pred_05.mean()))
print("val positive rate @best_thr:", float(val_pred_bt.mean()))
print("val class balance:", float(val_true.mean()))

In [ ]:
# Evaluate with tuned threshold
print("\nEvaluation with tuned threshold")
_ = evaluate_torch_model(model, val_features, val_targets, "Validation", device, threshold=best_thr)

In [ ]:
# AUC-PR driven hyperparameter search (model selection by validation AUC-PR)
from itertools import product

# Build test tensors (we only created train/val tensors in the original training cell)
test_features = torch.tensor(_test_array, dtype=torch.float32)
test_targets = torch.tensor(test_y.to_numpy(), dtype=torch.float32).unsqueeze(1)

base_pos_weight = neg / pos

search_space = {
    "hidden": [128, 256],
    "lr": [1e-3, 5e-4],
    "weight_decay": [1e-3, 1e-2],
    "pos_weight_scale": [0.5, 1.0],
}

# Keep sweep compact enough to run interactively in notebook
SWEEP_EPOCHS = 80
SWEEP_PATIENCE = 12


def train_one_config(cfg):
    model_cfg = MLP(input_size=train_features.shape[1], hidden=cfg["hidden"]).to(device)
    pw_value = base_pos_weight * cfg["pos_weight_scale"]
    criterion_cfg = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([pw_value], dtype=torch.float32, device=device)
    )
    optimizer_cfg = optim.AdamW(
        model_cfg.parameters(),
        lr=cfg["lr"],
        weight_decay=cfg["weight_decay"],
    )
    scheduler_cfg = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_cfg, mode="max", factor=0.5, patience=5, min_lr=1e-6
    )

    loader_cfg = DataLoader(train_dataset, batch_size=256, shuffle=True)

    best_aucpr = -1.0
    best_state_local = None
    no_imp = 0

    for _ in range(SWEEP_EPOCHS):
        model_cfg.train()
        for xb, yb in loader_cfg:
            xb, yb = xb.to(device), yb.to(device)
            loss_cfg = criterion_cfg(model_cfg(xb), yb)
            optimizer_cfg.zero_grad()
            loss_cfg.backward()
            optimizer_cfg.step()

        model_cfg.eval()
        with torch.no_grad():
            val_probs_cfg = torch.sigmoid(model_cfg(val_features.to(device)).squeeze(1)).cpu().numpy()
        val_true_cfg = val_targets.squeeze(1).detach().cpu().numpy().astype(int)
        val_aucpr_cfg = float(average_precision_score(val_true_cfg, val_probs_cfg))

        scheduler_cfg.step(val_aucpr_cfg)

        if val_aucpr_cfg > best_aucpr:
            best_aucpr = val_aucpr_cfg
            best_state_local = {k: v.detach().cpu().clone() for k, v in model_cfg.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1

        if no_imp >= SWEEP_PATIENCE:
            break

    model_cfg.load_state_dict(best_state_local)
    model_cfg.eval()
    with torch.no_grad():
        val_probs_cfg = torch.sigmoid(model_cfg(val_features.to(device)).squeeze(1)).cpu().numpy()

    # Keep threshold tuning for reporting only; selection remains AUC-PR based
    yv = val_targets.squeeze(1).detach().cpu().numpy().astype(int)
    p, r, t = precision_recall_curve(yv, val_probs_cfg)
    f1_curve = 2 * (p[:-1] * r[:-1]) / (p[:-1] + r[:-1] + 1e-12)
    idx = int(np.argmax(f1_curve))
    best_thr_cfg = float(t[idx])

    return {
        "cfg": cfg,
        "val_aucpr": best_aucpr,
        "val_best_f1": float(f1_curve[idx]),
        "best_thr": best_thr_cfg,
        "state": {k: v.detach().cpu().clone() for k, v in model_cfg.state_dict().items()},
    }


configs = [
    {
        "hidden": h,
        "lr": lr,
        "weight_decay": wd,
        "pos_weight_scale": pws,
    }
    for h, lr, wd, pws in product(
        search_space["hidden"],
        search_space["lr"],
        search_space["weight_decay"],
        search_space["pos_weight_scale"],
    )
]

results = []
for i, cfg in enumerate(configs, 1):
    res = train_one_config(cfg)
    results.append(res)
    print(
        f"[{i:>2}/{len(configs)}] cfg={cfg} | "
        f"val_AUC-PR={res['val_aucpr']:.4f} | "
        f"val_best_F1={res['val_best_f1']:.4f} | "
        f"thr={res['best_thr']:.4f}"
    )

best_result = max(results, key=lambda x: x["val_aucpr"])
print("\nBest config by validation AUC-PR:", best_result["cfg"])
print("Best validation AUC-PR:", round(best_result["val_aucpr"], 6))
print("Associated best-F1 threshold (reporting only):", round(best_result["best_thr"], 6))

# Promote best model to current `model` so later cells use it
model = MLP(input_size=train_features.shape[1], hidden=best_result["cfg"]["hidden"]).to(device)
model.load_state_dict(best_result["state"])
best_thr_aucpr = best_result["best_thr"]

print("\nFinal evaluation with AUC-PR-selected model")
train_metrics_aucpr = evaluate_torch_model(model, train_features, train_targets, "Train", device, threshold=best_thr_aucpr)
val_metrics_aucpr = evaluate_torch_model(model, val_features, val_targets, "Validation", device, threshold=best_thr_aucpr)

print("\nAUC-PR summary (selection objective)")
print({
    "train_auc_pr": float(train_metrics_aucpr["auc_pr"]),
    "val_auc_pr": float(val_metrics_aucpr["auc_pr"]),
})